# SpectralGuard Phase 3 — Automatic H100 Experiments
Select an H100 runtime, then click **Run all** once. The pipeline audits itself, resumes checkpoints, enforces a seven GPU-hour total cap, and creates `Experiments/exports/phase3_experiments.zip`. If a transient failure occurs, click **Run all** again: completed examples and attack restarts are not repeated.

In [ ]:
import importlib.util, subprocess, sys
if importlib.util.find_spec('pip') is None:
    subprocess.run([sys.executable, '-m', 'ensurepip', '--upgrade'], check=True)
%pip install -q -r requirements-lightning.txt
%pip install -q -e . --no-deps

In [ ]:
import importlib.metadata as md
import torch
assert md.version('torch') == '2.7.1', f'Expected torch 2.7.1, found {md.version("torch")}'
assert torch.cuda.is_available(), 'STOP: select an H100 runtime before Run all.'
gpu = torch.cuda.get_device_name(0)
assert 'H100' in gpu.upper(), f'STOP: H100 required, detected {gpu}'
assert torch.cuda.is_bf16_supported(), 'STOP: BF16 support is required.'
print({'torch': torch.__version__, 'cuda': torch.version.cuda, 'gpu': gpu, 'vram_gib': torch.cuda.get_device_properties(0).total_memory / 1024**3})

In [ ]:
from src.experiments.phase3_runner import run_automatic_phase3
automatic_report = run_automatic_phase3()
assert automatic_report['status'] == 'COMPLETE'
automatic_report

In [ ]:
from pathlib import Path
result = Path('Experiments/exports/phase3_experiments.zip').resolve()
assert result.is_file() and result.stat().st_size > 0
print(f'DOWNLOAD READY: {result}')
print('Next: stop the H100, open notebooks/02_phase3_analysis.ipynb on CPU, and click Run all.')